# 01 -- Datenexploration und Variablenauswahl

Dieses Notebook deckt die Tasks 1-4 des praktischen Analyseplans ab:

| Task | Beschreibung |
|------|-------------|
| **1** | Zielvariablen festlegen (HDI & Happiness Score) |
| **2** | Betrachtungszeitraum bestimmen |
| **3** | Betrachtete Laender bestimmen |
| **4** | Makrooekonomische Indikatoren auswaehlen |

**Datenquellen:**
- WISE Database (Liu et al., 2024): >1 Mio. Datenpunkte, 244 Metriken, 218 Laender
- Jones & Klenow (2016): lambda-Werte fuer 152 Laender (Referenzjahr 2007) -- wird in Notebook 04 verwendet

**Reproduzierbarkeit:** Alle Entscheidungen (Zeitraum, Laender, Indikatoren) werden in diesem Notebook
a priori dokumentiert und begruendet, bevor die Analyse stattfindet.

## 0 -- Setup

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Reproduzierbarkeit
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Pfade (relativ zum /code-Verzeichnis)
ROOT      = Path('..') / 'Pulled Databases'
WISE_PATH = ROOT / 'WISE_Database' / 'WISE_Database' / 'Data' / 'WISE_Database' / 'WISE_Database.xlsx'
JK_PATH   = ROOT / 'Jones_and_Klenow_Database' / 'BeyondGDP500.xls'

assert WISE_PATH.exists(), f'WISE-Datei nicht gefunden: {WISE_PATH}'
assert JK_PATH.exists(),   f'Jones/Klenow-Datei nicht gefunden: {JK_PATH}'

OUTPUT_DIR = Path('Created Files for Analysis')
OUTPUT_DIR.mkdir(exist_ok=True)

## 1 -- WISE Metadaten laden

Die `Metrics Info`-Sheet enthält alle Metadaten zu den 244 Metriken der WISE-Datenbank,
inkl. Kategorisierung (Wellbeing/Inclusion/Sustainability/Economy), Zeitraum und Laenderabdeckung.

In [3]:
metrics_info = pd.read_excel(WISE_PATH, sheet_name='Metrics Info')

print(f'Anzahl Metriken: {len(metrics_info)}')
print(f'Spalten: {list(metrics_info.columns)}')
metrics_info.head(3)

Anzahl Metriken: 244
Spalten: ['Wellbeing', 'Inclusion', 'Sustainability', 'Economy and Society', 'Subjective', 'Index', 'Inclusion Type', 'Theme', 'Measurement Type', 'Metric Full Name', 'Metric Acronym', 'Acronym', 'Unit', 'Tier', 'Metric Description', 'Data Source Full Name', 'Data Source Acronym', 'Reference', 'Source Link', 'Available Country Count', 'Start Year', 'End Year', 'Scale Min', 'Scale Max']


,Wellbeing,Inclusion,Sustainability,Economy and Society,Subjective,Index,Inclusion Type,Theme,Measurement Type,Metric Full Name,...,Metric Description,Data Source Full Name,Data Source Acronym,Reference,Source Link,Available Country Count,Start Year,End Year,Scale Min,Scale Max
0,X,NaN,NaN,NaN,NaN,X,NaN,Summary,Level,Composite Measure of Wellbeing (Historic Series),...,Composite measure of wellbeing. The composite ...,Clio-Infra (Composite Measure of Wellbeing),CLIO-CMW,"Rijpma, A. Composite measure of wellbeing. (20...",https://clio-infra.eu/Indicators/CompositeMeas...,193,1820,2000,-1.6,3.6
1,X,NaN,NaN,NaN,NaN,NaN,NaN,Education,Average per person,Average Years Of Education (Historic Series),...,Average years of education per country - Avera...,Clio-Infra (Composite Measure of Wellbeing),CLIO-CMW,"van Leeuwen, B. & van Leeuwen-Li, J. Average Y...",https://clio-infra.eu/Indicators/AverageYearso...,138,1870,2010,0.0,13.6
2,X,NaN,NaN,X,NaN,NaN,NaN,Income,Average per person,GDP Per Capita (Historic Series),...,Gross Domestic Product per Capita.,Clio-Infra (Composite Measure of Wellbeing),CLIO-CMW,"Bolt, J. & van Zanden, J. L. GDP per Capita. (...",https://clio-infra.eu/Indicators/GDPperCapita....,164,1950,2016,295.0,156299.0


## Task 1 -- Zielvariablen: HDI & Happiness Score

Die Arbeit verwendet zwei komplementäre Zielvariablen, die zusammen die objektive
und subjektive Seite des Wohlergehens abdecken (vgl. CMEPSP 2009)

### Übersicht aller HDI-verwandten Variablen in der WISE-Datenbank

Die folgende Tabelle listet alle Variablen, deren Akronym 'HDI' enthaelt oder
deren vollstaendiger Name 'Human Development Index' enthaelt.
Spalten: Variablenname (Akronym), Anzahl Laender, Zeitraum, Beschreibung.
Fehlende Beschreibungen ('—') bedeuten, dass die WISE-Datenbank fuer diese
Variable keinen Eintrag in der Spalte 'Metric Description' hat.

In [4]:
# Tabelle 1: Alle HDI-verwandten Variablen (direkt aus WISE Metrics Info)
hdi_vars = metrics_info[
    metrics_info['Acronym'].str.contains('HDI', na=False) |
    metrics_info['Metric Full Name'].str.contains('Human Development Index', case=False, na=False)
][['Metric Full Name', 'Acronym', 'Available Country Count', 'Start Year', 'End Year', 'Metric Description']].copy()

hdi_vars['Zeitraum'] = (
    hdi_vars['Start Year'].astype(int).astype(str) + '-' +
    hdi_vars['End Year'].astype(int).astype(str)
)
hdi_vars['Beschreibung'] = hdi_vars['Metric Description'].fillna('--')

hdi_display = hdi_vars[['Metric Full Name', 'Acronym', 'Available Country Count', 'Zeitraum', 'Beschreibung']].rename(columns={
    'Metric Full Name':        'Name',
    'Acronym':                 'Variablenname',
    'Available Country Count': 'Anzahl Laender',
}).reset_index(drop=True)

print(f'Anzahl HDI-verwandter Variablen in WISE: {len(hdi_display)}')
pd.set_option('display.max_colwidth', 300)
display(hdi_display)

Anzahl HDI-verwandter Variablen in WISE: 46


,Name,Variablenname,Anzahl Laender,Zeitraum,Beschreibung
0,Augmented Human Development Index,DLLP-AHDI,162,1870-2020,"Augmented Human Development Index (AHDI) combines achievements in health, education, material living standards, and political freedom."
1,Augmented Human Development Index (excluding the income dimension),DLLP-AHDI-AHDI*,162,1870-2020,"Augmented Human Development Index (AHDI) combines achievements in health, education, material living standards, and political freedom."
2,Population,DLLP-AHDI-POP,162,1870-2020,Population
3,Per Capita GDP (Geary-Khamis $1990),DLLP-AHDI-GDP-PC,162,1870-2020,"GDP per head is expressed in 1990 dollars adjusted for its purchasing power adjusted, that it, for the difference in price level across countries (the so-called Geary-Khamis [G-K] 1990 $)."
4,Kakwani Index of Life Expectancy,DLLP-AHDI-KLEB,162,1870-2020,"Augmented Human Development Index (AHDI) combines achievements in health, education, material living standards, and political freedom."
5,Kakwani Index of Schooling,DLLP-AHDI-KSCHL,162,1870-2020,"Augmented Human Development Index (AHDI) combines achievements in health, education, material living standards, and political freedom."
6,Liberal Democracy Index,DLLP-AHDI-LDI,162,1870-2020,"The Liberal Democracy Index combines the electoral democracy index and the liberal component index. The former incorporates indices of freedom of association, expression, suffrage, and clean elections. The latter includes indices of equality before the law and individual liberty, judicial constr..."
7,Life Expectancy at Birth,DLLP-AHDI-LEB,162,1870-2020,Life expectancy is defined as the average number of years of life which would remain for males and females reaching the ages specified if they continued to be subjected to the same mortality experienced in the year(s) to which these life expectancies refer.
8,Years of Schooling,DLLP-AHDI-SCHL,162,1870-2020,"Education attainment is measured by the average years of total schooling (primary, secondary, and tertiary) for the population aged 15 and over."
9,UNDP Adjusted Per Capita Income Index,DLLP-AHDI-UNDPY,162,1870-2020,"Augmented Human Development Index (AHDI) combines achievements in health, education, material living standards, and political freedom."


### Übersicht aller Life-Satisfaction-Variablen in der WISE-Datenbank

Die folgende Tabelle listet alle Variablen mit Theme 'Subjective Wellbeing'
oder Akronym beginnend mit 'WHR'.
Spalten: Variablenname (Akronym), Anzahl Laender, Zeitraum, Beschreibung.

In [5]:
# Tabelle 2: Alle LS / Subjective-Wellbeing-Variablen (direkt aus WISE Metrics Info)
ls_vars = metrics_info[
    (
        (metrics_info['Theme'] == 'Subjective Wellbeing') |
        (metrics_info['Acronym'].str.startswith('WHR', na=False))
    ) &
    (~metrics_info['Acronym'].str.startswith('FAN-DEF-SF-SS', na=False))
][['Metric Full Name', 'Acronym', 'Available Country Count', 'Start Year', 'End Year', 'Metric Description']].copy()

ls_vars['Zeitraum'] = (
    ls_vars['Start Year'].astype(int).astype(str) + '-' +
    ls_vars['End Year'].astype(int).astype(str)
)
ls_vars['Beschreibung'] = ls_vars['Metric Description'].fillna('--')

ls_display = ls_vars[['Metric Full Name', 'Acronym', 'Available Country Count', 'Zeitraum', 'Beschreibung']].rename(columns={
    'Metric Full Name':        'Name',
    'Acronym':                 'Variablenname',
    'Available Country Count': 'Anzahl Länder',
}).reset_index(drop=True)

print(f'Anzahl LS-verwandter Variablen in WISE: {len(ls_display)}')
display(ls_display)

Anzahl LS-verwandter Variablen in WISE: 12


,Name,Variablenname,Anzahl Länder,Zeitraum,Beschreibung
0,Social Foundation-Life Satisfaction-Value,FAN-DEF-SF-LS,119,2005-2015,"Responses to the life satisfaction questions of each survey, transformed using the Gallup World Poll’s Cantril life ladder question (0–10 scale) as a benchmark."
1,Social Foundation-Life Satisfaction-Ratio,FAN-DEF-SF-LS-RATIO,119,2005-2015,"Responses to the life satisfaction questions of each survey, transformed using the Gallup World Poll’s Cantril life ladder question (0–10 scale) as a benchmark."
2,Social Foundation-Life Satisfaction-ThresholdBoundary,FAN-DEF-SF-LS-THRES,148,1992-2015,"Responses to the life satisfaction questions of each survey, transformed using the Gallup World Poll’s Cantril life ladder question (0–10 scale) as a benchmark."
3,Life Satisfaction,WHR-LS,164,2006-2023,"Life evaluations. The Gallup World Poll, which remaIns the principal source of data in this report, asks respondents to evaluate their current life as a whole using the mental image of a ladder, with the best possible life for them as a 10 and worst possible as a 0. Each respondent provides a nu..."
4,Perceptions of corruption,WHR-LS-COR,159,2006-2023,"Perceptions of corruption are the average of binary answers to two GWP questions: “Is corruption widespread throughout the government in this country or not?” and “Is corruption widespread within businesses in this country or not?” Where data for government corruption are missing, the perception..."
5,Freedom to make life choices,WHR-LS-FR,164,2006-2023,"Freedom to make life choices is the national average of binary responses (0=no, 1=yes) to the GWP question “Are you satisfied or dissatisfied with your freedom to choose what you do with your life?”"
6,Generosity,WHR-LS-GE,162,2006-2023,Generosity is the residual of regressing the national average of GWP responses to the donation question “Have you donated money to a charity in the past month?” on log GDP per capita.
7,Healthy life expectancy at birth,WHR-LS-HALE,161,2006-2023,"The time series for healthy life expectancy at birth is constructed based on data from the World Health Organization (WHO) Global Health Observatory data repository, with data available for 2000, 2010, 2015, and 2019. Interpolation and extrapolation are used to match this report’s sample period ..."
8,Log GDP per capita,WHR-LS-LGDP,162,2006-2023,"GDP per capita is in terms of Purchasing Power Parity (PPP) adjusted to constant 2017 international dollars, taken from the World Development Indicators (WDI) released by the World Bank on December 16, 2021. See Statistical Appendix 1 for more details. GDP data for 2021 are not yet available, so..."
9,Negative affect,WHR-LS-NA,163,2006-2023,"Negative affect is defined as the average of previous-day affect measures for worry, sadness, and anger."


In [6]:
# Beide Tabellen in eine Excel-Datei mit zwei Sheets speichern
OUTPUT_FILE = OUTPUT_DIR / 'possible_target_vars.xlsx'

with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    hdi_display.to_excel(writer, sheet_name='HDI_Variables',  index=False)
    ls_display.to_excel( writer, sheet_name='LS_Variables',   index=False)

print(f'Gespeichert: {OUTPUT_FILE}')
print(f'  Sheet "HDI_Variables": {len(hdi_display)} Variablen')
print(f'  Sheet "LS_Variables":  {len(ls_display)} Variablen')

Gespeichert: Created Files for Analysis\possible_target_vars.xlsx
  Sheet "HDI_Variables": 46 Variablen
  Sheet "LS_Variables":  12 Variablen


### Variablen für kommende Analyse

| Variable | Akronym | Art | Quelle |
|----------|---------|-----|--------|
| Human Development Index | `UNDP-HDI` | Objektiv | UNDP |
| Life Satisfaction (Cantril-Ladder) | `WHR-LS` | Subjektiv | World Happiness Report / Gallup |

In [7]:
TARGET_ACRONYMS = ['UNDP-HDI', 'WHR-LS']

target_meta = metrics_info[
    metrics_info['Acronym'].isin(TARGET_ACRONYMS)
][[
    'Metric Full Name', 'Acronym', 'Unit', 'Tier',
    'Available Country Count', 'Start Year', 'End Year',
    'Wellbeing', 'Subjective', 'Index', 'Theme'
]].reset_index(drop=True)

print('=== Zielvariablen in der WISE-Datenbank ===')
display(target_meta)

hdi_meta = target_meta[target_meta['Acronym'] == 'UNDP-HDI'].iloc[0]
whr_meta = target_meta[target_meta['Acronym'] == 'WHR-LS'].iloc[0]

print(f"HDI:             {int(hdi_meta['Available Country Count'])} Laender, "
      f"{int(hdi_meta['Start Year'])}-{int(hdi_meta['End Year'])}")
print(f"Happiness Score: {int(whr_meta['Available Country Count'])} Laender, "
      f"{int(whr_meta['Start Year'])}-{int(whr_meta['End Year'])}")

=== Zielvariablen in der WISE-Datenbank ===


,Metric Full Name,Acronym,Unit,Tier,Available Country Count,Start Year,End Year,Wellbeing,Subjective,Index,Theme
0,Human Development Index,UNDP-HDI,Index (0-1),1st,193,1990,2022,X,NaN,X,Summary
1,Life Satisfaction,WHR-LS,Index(0-10),1st,164,2006,2023,X,X,X,Subjective Wellbeing


HDI:             193 Laender, 1990-2022
Happiness Score: 164 Laender, 2006-2023


## Task 2 -- Betrachtungszeitraum

Der Betrachtungszeitraum wird durch den engsten gemeinsamen Überlappungsbereich
beider Zielvariablen mit ausreichender Datenqualität bestimmt. Die Entscheidung erfolgt auf Basis der
Datenverfügbarkeit und wird durch die empirische Länderabdeckung je Jahr validiert.

In [8]:
# Nur die beiden Zielvariablen aus C Data laden
print('Lade Zielvariablen aus C Data für Datenqualitätsanalyse ...')
quality_raw = pd.read_excel(
    WISE_PATH,
    sheet_name='C Data',
    usecols=['ISO3', 'Acronym', 'Year', 'Value']
)
quality_raw = quality_raw[quality_raw['Acronym'].isin(TARGET_ACRONYMS)].copy()
quality_raw['Year'] = quality_raw['Year'].astype(int)
print(f'Zeilen geladen: {len(quality_raw):,}')

Lade Zielvariablen aus C Data für Datenqualitätsanalyse ...
Zeilen geladen: 8,171


In [9]:
# Für jede Variable: Anzahl eindeutiger Lnder insgesamt (Nenner)
total_countries = (
    quality_raw[quality_raw['Value'].notna()]
    .groupby('Acronym')['ISO3']
    .nunique()
    .rename('total_countries')
)

# Für jedes (Variable, Jahr): Anzahl Länder mit Datenpunkt (Zähler)
yearly_counts = (
    quality_raw[quality_raw['Value'].notna()]
    .groupby(['Acronym', 'Year'])['ISO3']
    .nunique()
    .reset_index(name='n_countries')
)

# Coverage in % berechnen
yearly_counts = yearly_counts.merge(total_countries, on='Acronym')
yearly_counts['coverage_pct'] = (
    yearly_counts['n_countries'] / yearly_counts['total_countries'] * 100
).round(1)

# Pivot für Anzeige
quality_table = yearly_counts.pivot(
    index='Year', columns='Acronym', values='coverage_pct'
).rename(columns={'UNDP-HDI': 'HDI_coverage_pct', 'WHR-LS': 'LS_coverage_pct'})

print('=== Datenabdeckung je Jahr (% der Länder mit Datenpunkt) ===')
print(f'Nenner HDI: {int(total_countries["UNDP-HDI"])} Länder insgesamt')
print(f'Nenner LS:  {int(total_countries["WHR-LS"])} Länder insgesamt')
print()
display(quality_table.T.style.format('{:.1f}%', na_rep='--').background_gradient(
    cmap='RdYlGn', vmin=0, vmax=100, axis=1
))

=== Datenabdeckung je Jahr (% der Länder mit Datenpunkt) ===
Nenner HDI: 193 Länder insgesamt
Nenner LS:  164 Länder insgesamt



Year,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
Acronym,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
HDI_coverage_pct,73.6%,73.6%,73.6%,73.6%,73.6%,79.3%,79.3%,79.3%,79.3%,81.9%,91.2%,91.2%,91.7%,92.7%,93.8%,97.4%,97.4%,97.9%,97.9%,97.9%,99.0%,99.5%,99.5%,99.5%,99.5%,99.5%,99.5%,99.5%,99.5%,99.5%,99.5%,99.5%,100.0%,--
LS_coverage_pct,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,16.5%,54.3%,62.2%,67.1%,69.5%,75.6%,89.0%,86.0%,82.9%,87.8%,86.6%,86.0%,89.6%,86.0%,87.2%,70.7%,74.4%,85.4%,84.1%


### ENTSCHEIDUNG TASK 2: Betrachtungszeitraum
 Begründung:
   - HDI verfügbar ab 1990; Happiness Score erst ab 2006.
   - Engster gemeinsamer Überlappungsbereich: 2006-2022 (17 Jahre).
   - Datenqualität für Happiness Score in 2006 jedoch nicht ausreichend.
   - Unabhängig von Analyseergebnissen festgelegt.

In [10]:
# Pivotieren: eine Zeile pro (Land, Jahr) mit HDI und Happiness_Score als Spalten
# quality_raw enthält alle Zeilen fuer UNDP-HDI und WHR-LS aus C Data
target_wide = quality_raw.pivot_table(
    index=['ISO3', 'Year'],
    columns='Acronym',
    values='Value'
).reset_index()
target_wide.columns.name = None
target_wide = target_wide.rename(columns={
    'UNDP-HDI': 'HDI',
    'WHR-LS':   'Happiness_Score'
})

print(f'target_wide: {target_wide.shape[0]:,} Länder-Jahr-Beobachtungen')
print(f'Spalten: {list(target_wide.columns)}')
target_wide.head()

target_wide: 5,984 Länder-Jahr-Beobachtungen
Spalten: ['ISO3', 'Year', 'HDI', 'Happiness_Score']


,ISO3,Year,HDI,Happiness_Score
0,AFG,1990,0.284,NaN
1,AFG,1991,0.292,NaN
2,AFG,1992,0.299,NaN
3,AFG,1993,0.307,NaN
4,AFG,1994,0.300,NaN


In [11]:
YEAR_START = 2007
YEAR_END   = 2022

target_period = target_wide[
    (target_wide['Year'] >= YEAR_START) &
    (target_wide['Year'] <= YEAR_END)
].copy()

print(f'Gewählter Zeitraum: {YEAR_START}-{YEAR_END} ({YEAR_END - YEAR_START + 1} Jahre)')

Gewählter Zeitraum: 2007-2022 (16 Jahre)


## Task 3 -- Betrachtete Länder

Ein Land wird aufgenommen, wenn für beide Zielvariablen (UNDP-HDI und WHR-LS)
im Betrachtungszeitraum 2007-2022 ein Mindestanteil an Jahren mit gültigen
Datenpunkten vorhanden ist.

In [12]:
# Betrachtungszeitraum (aus Task 2)
N_YEARS    = YEAR_END - YEAR_START + 1  # 16 Jahre
THRESHOLDS = [0.50, 0.60, 0.70, 0.75, 0.80]

# Für jedes Land: Anzahl Jahre, in denen BEIDE Zielvariablen gleichzeitig vorhanden sind
country_both = (
    target_period
    .assign(both=lambda df: df['HDI'].notna() & df['Happiness_Score'].notna())
    .groupby('ISO3')['both']
    .sum()
    .reset_index(name='years_both')
)

# Verteilung: Wie viele Laender haben genau x Jahre mit beiden Variablen vorhanden?
print(f'=== Verteilung: Anzahl Jahre mit BEIDEN Zielvariablen vorhanden ===')
print(f'    (Zeitraum {YEAR_START}-{YEAR_END}, max. {N_YEARS} Jahre)')
print()

dist = country_both['years_both'].value_counts().sort_index()
cumulative = 0
for yrs in range(N_YEARS + 1):
    cnt        = dist.get(yrs, 0)
    cumulative += cnt
    pct        = cnt / len(country_both) * 100
    cum_pct    = cumulative / len(country_both) * 100
    print(f'  {yrs:2d} Jahre: {cnt:3d} Länder ({pct:4.1f}%)  kumuliert: {cum_pct:5.1f}%')

=== Verteilung: Anzahl Jahre mit BEIDEN Zielvariablen vorhanden ===
    (Zeitraum 2007-2022, max. 16 Jahre)

   0 Jahre:  35 Länder (17.9%)  kumuliert:  17.9%
   1 Jahre:   4 Länder ( 2.1%)  kumuliert:  20.0%
   2 Jahre:   1 Länder ( 0.5%)  kumuliert:  20.5%
   3 Jahre:   1 Länder ( 0.5%)  kumuliert:  21.0%
   4 Jahre:   6 Länder ( 3.1%)  kumuliert:  24.1%
   5 Jahre:   5 Länder ( 2.6%)  kumuliert:  26.7%
   6 Jahre:   0 Länder ( 0.0%)  kumuliert:  26.7%
   7 Jahre:   3 Länder ( 1.5%)  kumuliert:  28.2%
   8 Jahre:   2 Länder ( 1.0%)  kumuliert:  29.2%
   9 Jahre:   3 Länder ( 1.5%)  kumuliert:  30.8%
  10 Jahre:   7 Länder ( 3.6%)  kumuliert:  34.4%
  11 Jahre:   9 Länder ( 4.6%)  kumuliert:  39.0%
  12 Jahre:   6 Länder ( 3.1%)  kumuliert:  42.1%
  13 Jahre:   8 Länder ( 4.1%)  kumuliert:  46.2%
  14 Jahre:  23 Länder (11.8%)  kumuliert:  57.9%
  15 Jahre:  29 Länder (14.9%)  kumuliert:  72.8%
  16 Jahre:  53 Länder (27.2%)  kumuliert: 100.0%


### ENTSCHEIDUNG TASK 3: Threshold

Gewählt: **MIN_YEARS = 12** (mind. 12 von 16 Jahren mit beiden Zielvariablen vorhanden)

Entspricht 75% des Betrachtungszeitraums.

39% Der Länder werden sommit ausgeschlossen.

In [13]:
# Länderliste: mind. MIN_YEARS Jahre mit beiden Zielvariablen
MIN_YEARS = 12
INCLUDED_COUNTRIES = sorted(
    country_both.loc[
        country_both['years_both'] >= MIN_YEARS, 'ISO3'
    ].tolist()
)

EXCLUDED_COUNTRIES = sorted(
    country_both.loc[
        country_both['years_both'] < MIN_YEARS, 'ISO3'
    ].tolist()
)

print(f'Eingeschlossene Laender (years_both >= {MIN_YEARS}): {len(INCLUDED_COUNTRIES)}')
print(f'Ausgeschlossene Laender (years_both <  {MIN_YEARS}): {len(EXCLUDED_COUNTRIES)}')
print()
print('INCLUDED_COUNTRIES:')
print(INCLUDED_COUNTRIES)
print()
print('EXCLUDED_COUNTRIES:')
print(EXCLUDED_COUNTRIES)

Eingeschlossene Laender (years_both >= 12): 119
Ausgeschlossene Laender (years_both <  12): 76

INCLUDED_COUNTRIES:
['AFG', 'ALB', 'ARE', 'ARG', 'ARM', 'AUS', 'AUT', 'AZE', 'BEL', 'BEN', 'BFA', 'BGD', 'BGR', 'BIH', 'BLR', 'BOL', 'BRA', 'BWA', 'CAN', 'CHL', 'CHN', 'CMR', 'COG', 'COL', 'CRI', 'CYP', 'CZE', 'DEU', 'DNK', 'DOM', 'ECU', 'EGY', 'ESP', 'EST', 'FIN', 'FRA', 'GAB', 'GBR', 'GEO', 'GHA', 'GIN', 'GRC', 'GTM', 'HKG', 'HND', 'HRV', 'HUN', 'IDN', 'IND', 'IRL', 'IRN', 'IRQ', 'ISR', 'ITA', 'JOR', 'JPN', 'KAZ', 'KEN', 'KGZ', 'KHM', 'KOR', 'LBN', 'LKA', 'LTU', 'LUX', 'LVA', 'MAR', 'MDA', 'MEX', 'MKD', 'MLI', 'MLT', 'MNE', 'MNG', 'MRT', 'MWI', 'MYS', 'NER', 'NGA', 'NIC', 'NLD', 'NPL', 'NZL', 'PAK', 'PAN', 'PER', 'PHL', 'POL', 'PRT', 'PRY', 'PSE', 'ROU', 'RUS', 'SAU', 'SEN', 'SGP', 'SLE', 'SLV', 'SRB', 'SVK', 'SVN', 'SWE', 'TCD', 'THA', 'TJK', 'TUN', 'TUR', 'TZA', 'UGA', 'UKR', 'URY', 'USA', 'UZB', 'VEN', 'VNM', 'YEM', 'ZAF', 'ZMB', 'ZWE']

EXCLUDED_COUNTRIES:
['AGO', 'AND', 'ATG', 'BDI', 

## Task 4 -- Makroökonomische Indikatoren auswählen

Als Grundlage für die Auswahl der Indikatoren wird zunächst eine Übersicht
aller in der WISE-Datenbank enthaltenen Features erstellt. Die Übersicht enthält
alle verfügbaren Metadaten zu jedem Feature direkt aus der Datenbank.

Die Zielvariablen (UNDP-HDI, WHR-LS) sowie deren Sub-Indikatoren werden
ausgeklammert, da sie als abhängige Variablen dienen.

In [14]:
# Alle Features der WISE-Datenbank mit Metadaten
# Keine Quellenausschlüsse -- HDI- und LS-Sub-Indikatoren koennen
# als unabhaengige Praediktoren relevant sein

EXPORT_COLS = [
    'Metric Full Name',
    'Acronym',
    'Theme',
    'Tier',
    'Wellbeing',
    'Inclusion',
    'Sustainability',
    'Economy and Society',
    'Subjective',
    'Index',
    'Unit',
    'Measurement Type',
    'Available Country Count',
    'Start Year',
    'End Year',
    'Data Source Full Name',
    'Data Source Acronym',
    'Metric Description',
    'Reference',
]

# Alle Features
features_all = metrics_info[EXPORT_COLS].reset_index(drop=True)

# Nur Features aus HDI-Quelle (Sub-Indikatoren des UNDP-HDI)
features_hdi = features_all[
    features_all['Data Source Acronym'] == 'UNDP-HDI'
].reset_index(drop=True)

# Nur Features aus WHR-LS-Quelle (Sub-Indikatoren des WHR)
features_ls = features_all[
    features_all['Data Source Acronym'] == 'WHR-LS'
].reset_index(drop=True)

# Alle uebrigen Features (weder UNDP-HDI noch WHR-LS als Quelle)
features_other = features_all[
    ~features_all['Data Source Acronym'].isin(['UNDP-HDI', 'WHR-LS'])
].reset_index(drop=True)

print(f'Features gesamt:          {len(features_all)}')
print(f'  davon UNDP-HDI-Quelle:  {len(features_hdi)}')
print(f'  davon WHR-LS-Quelle:    {len(features_ls)}')
print(f'  davon andere Quellen:   {len(features_other)}')

Features gesamt:          244
  davon UNDP-HDI-Quelle:  36
  davon WHR-LS-Quelle:    9
  davon andere Quellen:   199


In [15]:
OUTPUT_FEATURES = OUTPUT_DIR / 'wise_features_overview.xlsx'

with pd.ExcelWriter(OUTPUT_FEATURES, engine='openpyxl') as writer:

    # Tab 1: Alle Features
    features_all.to_excel(writer, sheet_name='All_Features', index=False)

    # Tab 2: Nur 1st-Tier (Hauptindikatoren aller Quellen)
    features_all[
        features_all['Tier'] == '1st'
    ].to_excel(writer, sheet_name='1st_Tier_Only', index=False)

    # Tab 3: Alle uebrigen Features (ohne HDI- und LS-Quellen), nach Theme sortiert
    features_other.sort_values(
        ['Theme', 'Tier', 'Available Country Count'],
        ascending=[True, True, False]
    ).to_excel(writer, sheet_name='Other_Sources_By_Theme', index=False)

    # Tab 4: UNDP-HDI Sub-Indikatoren
    features_hdi.to_excel(writer, sheet_name='UNDP_HDI_Features', index=False)

    # Tab 5: WHR-LS Sub-Indikatoren
    features_ls.to_excel(writer, sheet_name='WHR_LS_Features', index=False)

print(f'Gespeichert: {OUTPUT_FEATURES}')
print(f'  Tab "All_Features":           {len(features_all)} Features')
print(f'  Tab "1st_Tier_Only":           {(features_all["Tier"] == "1st").sum()} Features')
print(f'  Tab "Other_Sources_By_Theme":  {len(features_other)} Features')
print(f'  Tab "UNDP_HDI_Features":       {len(features_hdi)} Features')
print(f'  Tab "WHR_LS_Features":         {len(features_ls)} Features')

Gespeichert: Created Files for Analysis\wise_features_overview.xlsx
  Tab "All_Features":           244 Features
  Tab "1st_Tier_Only":           135 Features
  Tab "Other_Sources_By_Theme":  199 Features
  Tab "UNDP_HDI_Features":       36 Features
  Tab "WHR_LS_Features":         9 Features


### Keyword-basierte Identifikation makroökonomischer Indikatoren

Die WISE-Kategorisierung (Theme, Tier) erfasst nicht alle makrooekonomischen
Indikatoren zuverlaessig -- z.B. liegen GNI pro Kopf oder Einkommensungleichheit
unter der UNDP-HDI-Quelle. Daher wird eine **Keyword-Suche ueber alle
Variablennamen und Beschreibungen** durchgefuehrt, unabhaengig von Quelle oder
Theme.

Fuer jedes Keyword wird sowohl `Metric Full Name` als auch `Metric Description`
durchsucht (case-insensitive). Das Ergebnis wird zur manuellen Nachpruefung
exportiert.

In [16]:
# -----------------------------------------------------------------------
# Keyword-Dictionary: Kategorie -> Liste von Suchbegriffen
# Suche laeuft ueber 'Metric Full Name' UND 'Metric Description'
# -----------------------------------------------------------------------
MACRO_KEYWORDS = {
    'Income / Output': [
        'gdp', 'gnp', 'gni', 'gross domestic', 'gross national',
        'national income', 'value added', 'output', 'purchasing power',
        'ppp', 'per capita income', 'real income', 'disposable income',
    ],
    'Labour / Employment': [
        'employment', 'unemployment', 'labour force', 'labor force',
        'workforce', 'working age', 'wage', 'salary', 'earnings',
        'job', 'work hours', 'productivity',
    ],
    'Trade': [
        'export', 'import', 'trade', 'current account',
        'balance of payment', 'merchandise', 'tariff',
    ],
    'Capital / Finance': [
        'investment', 'capital', 'fdi', 'foreign direct',
        'debt', 'deficit', 'surplus', 'credit', 'asset',
        'liability', 'net foreign', 'savings', 'financial',
    ],
    'Consumption / Expenditure': [
        'consumption', 'expenditure', 'spending',
        'government spending', 'household spending',
    ],
    'Inequality / Poverty': [
        'inequality', 'gini', 'poverty', 'distribution',
        'quintile', 'decile', 'income share', 'bottom',
        'wealth gap', 'deprivation',
    ],
    'Prices / Monetary': [
        'inflation', 'price', 'cpi', 'deflator',
        'monetary', 'interest rate', 'exchange rate',
    ],
}

print('Keyword-Kategorien:')
for cat, kws in MACRO_KEYWORDS.items():
    print(f'  {cat}: {len(kws)} Keywords')

Keyword-Kategorien:
  Income / Output: 13 Keywords
  Labour / Employment: 12 Keywords
  Trade: 7 Keywords
  Capital / Finance: 13 Keywords
  Consumption / Expenditure: 5 Keywords
  Inequality / Poverty: 10 Keywords
  Prices / Monetary: 7 Keywords


In [17]:
# Suchtext: Kombination aus Name und Beschreibung (lowercase)
all_features = metrics_info[[
    'Metric Full Name', 'Acronym', 'Theme', 'Tier',
    'Wellbeing', 'Inclusion', 'Sustainability', 'Economy and Society',
    'Subjective', 'Index', 'Unit', 'Measurement Type',
    'Available Country Count', 'Start Year', 'End Year',
    'Data Source Full Name', 'Data Source Acronym',
    'Metric Description', 'Reference',
]].copy()

search_text = (
    all_features['Metric Full Name'].fillna('').str.lower() + ' ' +
    all_features['Metric Description'].fillna('').str.lower()
)

# Fuer jede Variable: welche Keywords und Kategorien treffen zu?
def find_matches(text):
    matched_cats = []
    matched_kws  = []
    for cat, keywords in MACRO_KEYWORDS.items():
        hits = [kw for kw in keywords if kw in text]
        if hits:
            matched_cats.append(cat)
            matched_kws.extend(hits)
    return matched_cats, matched_kws

results = search_text.apply(find_matches)
all_features['matched_categories'] = results.apply(lambda x: ', '.join(x[0]))
all_features['matched_keywords']   = results.apply(lambda x: ', '.join(sorted(set(x[1]))))
all_features['n_keyword_matches']  = results.apply(lambda x: len(set(x[1])))
all_features['is_macro_candidate'] = all_features['n_keyword_matches'] > 0

# Zielvariablen selbst ausschliessen
all_features['is_target_var'] = all_features['Acronym'].isin(['UNDP-HDI', 'WHR-LS'])

candidates = all_features[
    all_features['is_macro_candidate'] & ~all_features['is_target_var']
].sort_values(
    ['n_keyword_matches', 'Tier'],
    ascending=[False, True]
).reset_index(drop=True)

non_candidates = all_features[
    ~all_features['is_macro_candidate'] & ~all_features['is_target_var']
].sort_values(['Theme', 'Tier']).reset_index(drop=True)

print(f'Gesamtzahl Features (ohne Zielvariablen): {len(all_features) - 2}')
print(f'  Makrooekonomische Kandidaten:           {len(candidates)}')
print(f'  Kein Keyword-Treffer:                   {len(non_candidates)}')
print()
print('Kandidaten nach Kategorie:')
for cat in MACRO_KEYWORDS:
    n = candidates['matched_categories'].str.contains(cat, regex=False).sum()
    print(f'  {cat}: {n}')

Gesamtzahl Features (ohne Zielvariablen): 242
  Makrooekonomische Kandidaten:           134
  Kein Keyword-Treffer:                   108

Kandidaten nach Kategorie:
  Income / Output: 24
  Labour / Employment: 16
  Trade: 7
  Capital / Finance: 65
  Consumption / Expenditure: 26
  Inequality / Poverty: 21
  Prices / Monetary: 7


In [18]:
OUTPUT_CANDIDATES = OUTPUT_DIR / 'macro_indicator_candidates.xlsx'

with pd.ExcelWriter(OUTPUT_CANDIDATES, engine='openpyxl') as writer:

    # Tab 1: Alle Kandidaten (Keyword-Treffer), sortiert nach Anzahl Matches
    candidates.drop(columns=['is_macro_candidate', 'is_target_var']).to_excel(
        writer, sheet_name='Candidates', index=False
    )

    # Tab 2: Kandidaten -- nur 1st Tier
    candidates[
        candidates['Tier'] == '1st'
    ].drop(columns=['is_macro_candidate', 'is_target_var']).to_excel(
        writer, sheet_name='Candidates_1st_Tier', index=False
    )

    # Tab 3: Kein Keyword-Treffer (zur manuellen Pruefung auf Luecken)
    non_candidates.drop(columns=['is_macro_candidate', 'is_target_var']).to_excel(
        writer, sheet_name='No_Match_Review', index=False
    )

    # Tab 4: Alle Features mit Match-Spalten (vollstaendige Uebersicht)
    all_features[
        ~all_features['is_target_var']
    ].sort_values(
        ['is_macro_candidate', 'n_keyword_matches'],
        ascending=[False, False]
    ).drop(columns=['is_macro_candidate', 'is_target_var']).to_excel(
        writer, sheet_name='All_With_Match_Info', index=False
    )

print(f'Gespeichert: {OUTPUT_CANDIDATES}')
print(f'  Tab "Candidates":          {len(candidates)} Eintraege')
print(f'  Tab "Candidates_1st_Tier": '
      f'{(candidates["Tier"] == "1st").sum()} Eintraege')
print(f'  Tab "No_Match_Review":     {len(non_candidates)} Eintraege')
print(f'  Tab "All_With_Match_Info": {len(all_features) - 2} Eintraege')

Gespeichert: Created Files for Analysis\macro_indicator_candidates.xlsx
  Tab "Candidates":          134 Eintraege
  Tab "Candidates_1st_Tier": 64 Eintraege
  Tab "No_Match_Review":     108 Eintraege
  Tab "All_With_Match_Info": 242 Eintraege


### Datenqualität der ausgewählten makroökonomischen Indikatoren

Fuer jeden der manuell ausgewählten Indikatoren wird geprueft:
- **Länderabdeckung (%):** Anteil der `INCLUDED_COUNTRIES`, für die im
  Zeitraum 2007-2022 mindestens ein Datenpunkt vorliegt
- **Jahresabdeckung (%):** Durchschnittlicher Anteil der Jahre 2007-2022,
  für die pro Land ein Datenpunkt vorliegt (gemittelt über `INCLUDED_COUNTRIES`)

Grundlage ist die Datei `makrooekonomische_indikatoren.xlsx`.

In [19]:
# Pfad zur Indikatorenliste
INDICATORS_PATH = Path('..') / '..' / 'makrooekonomische_indikatoren.xlsx'

# Indikatoren laden (Zeile 0 = Header 'Name/Beschreibung/Begruendung')
ind_df = pd.read_excel(INDICATORS_PATH, header=1)
ind_df.columns = ['Name', 'Beschreibung', 'Begruendung']
ind_df = ind_df.dropna(subset=['Name']).reset_index(drop=True)

# Akronyme aus WISE-Metadaten ergaenzen (exakter Name-Match)
name_to_acronym = metrics_info.set_index('Metric Full Name')['Acronym'].to_dict()
ind_df['Acronym'] = ind_df['Name'].str.strip().map(name_to_acronym)

not_matched = ind_df[ind_df['Acronym'].isna()]['Name'].tolist()
print(f'Indikatoren geladen:    {len(ind_df)}')
print(f'Davon nicht gematcht:   {len(not_matched)}')
if not_matched:
    print('Nicht gefunden:', not_matched)

Indikatoren geladen:    38
Davon nicht gematcht:   0


In [20]:
# C Data fuer alle Kandidaten-Akronyme laden
INDICATOR_ACRONYMS = ind_df['Acronym'].dropna().tolist()

print('Lade C Data fuer Indikator-Coverage ...')
c_ind = pd.read_excel(
    WISE_PATH, sheet_name='C Data',
    usecols=['ISO3', 'Acronym', 'Year', 'Value']
)
c_ind['Year'] = c_ind['Year'].astype(int)

# Auf Zeitraum und INCLUDED_COUNTRIES einschraenken
c_ind = c_ind[
    c_ind['Acronym'].isin(INDICATOR_ACRONYMS) &
    c_ind['ISO3'].isin(INCLUDED_COUNTRIES) &
    c_ind['Year'].between(YEAR_START, YEAR_END) &
    c_ind['Value'].notna()
].copy()

n_countries = len(INCLUDED_COUNTRIES)
n_years     = N_YEARS  # 16

rows = []
for _, row in ind_df.iterrows():
    acr  = row['Acronym']
    sub  = c_ind[c_ind['Acronym'] == acr] if acr and not pd.isna(acr) else pd.DataFrame()

    # Laenderabdeckung: Anteil INCLUDED_COUNTRIES mit mind. 1 Datenpunkt
    countries_with_data = sub['ISO3'].nunique()
    country_pct         = round(countries_with_data / n_countries * 100, 1)

    # Jahresabdeckung: mittlerer Anteil Jahre mit Datenpunkt je Land
    if len(sub) > 0:
        years_per_country = sub.groupby('ISO3')['Year'].nunique()
        # Laender ohne jeglichen Datenpunkt erhalten 0 Jahre
        all_zeros = pd.Series(0, index=[
            c for c in INCLUDED_COUNTRIES if c not in years_per_country.index
        ])
        years_per_country = pd.concat([years_per_country, all_zeros])
        year_pct = round(years_per_country.mean() / n_years * 100, 1)
    else:
        year_pct = 0.0

    rows.append({
        'Acronym':              acr,
        'Name':                 row['Name'],
        'Beschreibung':         row['Beschreibung'],
        'Begruendung':          row['Begruendung'],
        'Laender_mit_Daten':    countries_with_data,
        'Laenderabdeckung_pct': country_pct,
        'Jahresabdeckung_pct':  year_pct,
    })

coverage_df = pd.DataFrame(rows).sort_values(
    ['Laenderabdeckung_pct', 'Jahresabdeckung_pct'],
    ascending=False
).reset_index(drop=True)

print(f'Coverage berechnet fuer {len(coverage_df)} Indikatoren')
print(f'Basis: {n_countries} Laender, {n_years} Jahre ({YEAR_START}-{YEAR_END})')
print()
display(coverage_df[[
    'Acronym', 'Name', 'Laenderabdeckung_pct', 'Jahresabdeckung_pct'
]].style.format({
    'Laenderabdeckung_pct': '{:.1f}%',
    'Jahresabdeckung_pct':  '{:.1f}%',
}).background_gradient(cmap='RdYlGn', subset=[
    'Laenderabdeckung_pct', 'Jahresabdeckung_pct'
], vmin=0, vmax=100))

Lade C Data fuer Indikator-Coverage ...
Coverage berechnet fuer 38 Indikatoren
Basis: 119 Laender, 16 Jahre (2007-2022)



,Acronym,Name,Laenderabdeckung_pct,Jahresabdeckung_pct
0,UNDP-HDI-GNIPC,Gross national income (GNI) per capita,100.0%,100.0%
1,WB-WDI-SL-TLF-TOTL-IN,"Labor force, total",100.0%,99.9%
2,UNDP-HDI-GDI-GNI-PC-F,Estimated gross national income per capita (Female),100.0%,98.7%
3,UNDP-HDI-GDI-GNI-PC-M,Estimated gross national income per capita (Male),100.0%,98.7%
4,UNDP-HDI-GII-LFPR-F,Labour force participation rate (Female),100.0%,98.7%
5,UNDP-HDI-GII-LFPR-M,Labour force participation rate (Male),100.0%,98.7%
6,WB-WDI-FM-AST-NFRG-CN,Net foreign assets,100.0%,94.6%
7,WB-WDI-EG-EGY-PRIM-PP-KD,Energy intensity level of primary energy,100.0%,90.0%
8,WB-WDI-SL-UEM-TOTL-NE-ZS,"Unemployment, total (% of total labor force)",100.0%,79.3%
9,WB-WDI-SL-EMP-TOTL-SP-NE-ZS,"Employment to population ratio, 15+, total (%)",100.0%,77.8%


In [21]:
OUTPUT_COVERAGE = OUTPUT_DIR / 'indicator_coverage.xlsx'

with pd.ExcelWriter(OUTPUT_COVERAGE, engine='openpyxl') as writer:

    # Tab 1: Alle Indikatoren sortiert nach Abdeckung
    coverage_df.to_excel(writer, sheet_name='Coverage', index=False)

    # Tab 2: Nur Indikatoren mit vollstaendiger Laenderabdeckung (100%)
    coverage_df[
        coverage_df['Laenderabdeckung_pct'] == 100.0
    ].to_excel(writer, sheet_name='Full_Country_Coverage', index=False)

    # Tab 3: Indikatoren mit Laenderabdeckung < 80% (potenzielle Problemfaelle)
    coverage_df[
        coverage_df['Laenderabdeckung_pct'] < 80.0
    ].to_excel(writer, sheet_name='Below_80pct_Country', index=False)

print(f'Gespeichert: {OUTPUT_COVERAGE}')
print(f'  Tab "Coverage":              {len(coverage_df)} Indikatoren')
print(f'  Tab "Full_Country_Coverage": '
      f'{(coverage_df["Laenderabdeckung_pct"] == 100.0).sum()} Indikatoren')
print(f'  Tab "Below_80pct_Country":   '
      f'{(coverage_df["Laenderabdeckung_pct"] < 80.0).sum()} Indikatoren')

Gespeichert: Created Files for Analysis\indicator_coverage.xlsx
  Tab "Coverage":              38 Indikatoren
  Tab "Full_Country_Coverage": 10 Indikatoren
  Tab "Below_80pct_Country":   4 Indikatoren


In [22]:
# Features entfernen, die MINDESTENS EINE der Bedingungen erfuellen:
#   - Laenderabdeckung < 80%  ODER
#   - Jahresabdeckung  < 75%
# Ausnahmen: folgende Features werden unabhaengig davon behalten
KEEP_ALWAYS = [
    'Net foreign assets per capita',
    'Produced capital per capita',
    'Employment to population ratio, ages 15-24, total (%)',
]

# Inhaltlicher Ausschluss unabhaengig von der Coverage:
#   - Net foreign assets (WB-WDI-FM-AST-NFRG-CN): Einheit 'current LCU'
#     (nominale Landeswaehrung) -- ueber Laender nicht vergleichbar.
#     (Nicht zu verwechseln mit 'Net foreign assets per capita' (CWON), das bleibt.)
DROP_ALWAYS = [
    'Net foreign assets',
]

MIN_COUNTRY_PCT = 80.0
MIN_YEAR_PCT    = 75.0

remove_mask = (
    (
        (
            (coverage_df['Laenderabdeckung_pct'] < MIN_COUNTRY_PCT) |
            (coverage_df['Jahresabdeckung_pct']  < MIN_YEAR_PCT)
        ) &
        (~coverage_df['Name'].isin(KEEP_ALWAYS))
    ) |
    coverage_df['Name'].isin(DROP_ALWAYS)
)

removed  = coverage_df[remove_mask].reset_index(drop=True)
final_df = coverage_df[~remove_mask].reset_index(drop=True)

print(f'Indikatoren vor Filter:   {len(coverage_df)}')
print(f'Entfernt:                 {len(removed)}')
print(f'Verbleibend:              {len(final_df)}')
print()
if len(removed) > 0:
    print('=== Entfernte Indikatoren ===')
    for _, row in removed.iterrows():
        print(f'  [{row["Acronym"]}] {row["Name"]} '
              f'(Laender: {row["Laenderabdeckung_pct"]}%, '
              f'Jahre: {row["Jahresabdeckung_pct"]}%)')
print()
print('=== Beibehaltene Indikatoren (KEEP_ALWAYS, auch wenn unter Schwelle) ===')
for name in KEEP_ALWAYS:
    row = coverage_df[coverage_df['Name'] == name]
    if len(row) > 0:
        r = row.iloc[0]
        print(f'  [{r["Acronym"]}] {r["Name"]} '
              f'(Laender: {r["Laenderabdeckung_pct"]}%, '
              f'Jahre: {r["Jahresabdeckung_pct"]}%)')
print()

# Finale Liste speichern
OUTPUT_FINAL = OUTPUT_DIR / 'selected_indicators_final.xlsx'
final_df.to_excel(OUTPUT_FINAL, index=False)
print(f'Finale Indikatorenliste gespeichert: {OUTPUT_FINAL}')
print(f'Enthaltene Akronyme:')
print(final_df['Acronym'].tolist())

Indikatoren vor Filter:   38
Entfernt:                 12
Verbleibend:              26

=== Entfernte Indikatoren ===
  [WB-WDI-SI-POV-DDAY] Poverty headcount ratio at $2.15 a day (2017 PPP) (Laender: 93.3%, Jahre: 55.0%)
  [WB-WDI-SI-POV-GINI] Gini index (Laender: 93.3%, Jahre: 55.0%)
  [WB-WDI-SI-DST-50MD] Proportion of people living below 50 percent of median income (Laender: 93.3%, Jahre: 54.9%)
  [WB-WDI-EG-IMP-CONS-ZS] Energy imports, net (Laender: 91.6%, Jahre: 47.1%)
  [WB-WDI-EG-GDP-PUSE-KO-PP-KD] GDP per unit of energy use (Laender: 89.9%, Jahre: 46.4%)
  [WB-WDI-GB-XPD-RSDV-GD-ZS] Research and development expenditure (Laender: 89.1%, Jahre: 62.3%)
  [WB-WDI-GC-NFN-TOTL-CN] Net investment in nonfinancial assets (Laender: 85.7%, Jahre: 46.8%)
  [WB-WDI-SH-UHC-NOP2-ZS] Proportion of population pushed below the $3.65 ($ 2017 PPP) poverty line by out-of-pocket health care expenditure (Laender: 81.5%, Jahre: 24.3%)
  [WB-WDI-TM-VAL-MRCH-WR-ZS] Merchandise imports from low- and mid

## Export: Konfiguration und Rohdaten für andere Notebooks

Alle Entscheidungen aus Tasks 1-4 werden als `config.json` gespeichert.
Die zugehoerigen Rohdaten aus der WISE-Datenbank werden als `wise_data_raw.csv`
exportiert, sodass Notebook 02 die grosse WISE-Datei nicht mehr laden muss.

**Inhalt `config.json`:** Zielvariablen, Zeitraum, Laenderliste, Indikator-Akronyme,
Metadaten zu jedem Indikator.

**Inhalt `wise_data_raw.csv`:** Long-Format (ISO3, Year, Acronym, Value) fuer alle
Zielvariablen und ausgewaehlten Indikatoren, gefiltert auf INCLUDED_COUNTRIES
und den Betrachtungszeitraum.

In [23]:
import json

# Akronyme der finalen Indikatoren
SELECTED_INDICATOR_ACRONYMS = final_df['Acronym'].dropna().tolist()

# Metadaten je Indikator (Name, Beschreibung, Begruendung aus Nutzerliste)
indicator_meta = final_df[[
    'Acronym', 'Name', 'Beschreibung', 'Begruendung',
    'Laenderabdeckung_pct', 'Jahresabdeckung_pct'
]].to_dict(orient='records')

config = {
    # Zielvariablen
    'target_acronyms': TARGET_ACRONYMS,
    'target_hdi':      'UNDP-HDI',
    'target_hs':       'WHR-LS',

    # Betrachtungszeitraum
    'year_start': YEAR_START,
    'year_end':   YEAR_END,
    'n_years':    N_YEARS,

    # Laender
    'included_countries':    INCLUDED_COUNTRIES,
    'n_countries':           len(INCLUDED_COUNTRIES),
    'min_years_both':        MIN_YEARS,

    # Indikatoren
    'selected_indicator_acronyms': SELECTED_INDICATOR_ACRONYMS,
    'n_indicators':                len(SELECTED_INDICATOR_ACRONYMS),
    'indicator_meta':              indicator_meta,

    # Alle Akronyme (Zielvariablen + Indikatoren)
    'all_acronyms': TARGET_ACRONYMS + SELECTED_INDICATOR_ACRONYMS,
}

with open(OUTPUT_DIR / 'config.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print('config.json gespeichert')
print(f'  Zielvariablen:  {TARGET_ACRONYMS}')
print(f'  Zeitraum:       {YEAR_START}-{YEAR_END} ({N_YEARS} Jahre)')
print(f'  Laender:        {len(INCLUDED_COUNTRIES)}')
print(f'  Indikatoren:    {len(SELECTED_INDICATOR_ACRONYMS)}')
print(f'  Akronyme gesamt:{len(config["all_acronyms"])}')

config.json gespeichert
  Zielvariablen:  ['UNDP-HDI', 'WHR-LS']
  Zeitraum:       2007-2022 (16 Jahre)
  Laender:        119
  Indikatoren:    26
  Akronyme gesamt:28


In [24]:
# Rohdaten aus WISE für alle relevanten Akronyme, Laender und Jahre laden
ALL_ACRONYMS = TARGET_ACRONYMS + SELECTED_INDICATOR_ACRONYMS

print('Lade WISE C Data ...')
c_full = pd.read_excel(
    WISE_PATH, sheet_name='C Data',
    usecols=['ISO3', 'Acronym', 'Year', 'Value']
)
c_full['Year'] = c_full['Year'].astype(int)

wise_raw = c_full[
    c_full['Acronym'].isin(ALL_ACRONYMS) &
    c_full['ISO3'].isin(INCLUDED_COUNTRIES) &
    c_full['Year'].between(YEAR_START, YEAR_END)
].copy().reset_index(drop=True)

# Metadaten-Spalten ergaenzen (Name, Einheit)
acronym_to_name = metrics_info.set_index('Acronym')['Metric Full Name'].to_dict()
acronym_to_unit = metrics_info.set_index('Acronym')['Unit'].to_dict()
wise_raw['Metric_Name'] = wise_raw['Acronym'].map(acronym_to_name)
wise_raw['Unit']        = wise_raw['Acronym'].map(acronym_to_unit)

# Spaltenreihenfolge
wise_raw = wise_raw[['ISO3', 'Year', 'Acronym', 'Metric_Name', 'Unit', 'Value']]

OUTPUT_RAW = OUTPUT_DIR / 'wise_data_raw.csv'
wise_raw.to_csv(OUTPUT_RAW, index=False, encoding='utf-8')

print(f'wise_data_raw.csv gespeichert')
print(f'  Zeilen gesamt:   {len(wise_raw):,}')
print(f'  Akronyme:        {wise_raw["Acronym"].nunique()}')
print(f'  Laender:         {wise_raw["ISO3"].nunique()}')
print(f'  Jahre:           {sorted(wise_raw["Year"].unique())}')
print()
print('Zeilen je Akronym:')
print(wise_raw.groupby('Acronym')['Value'].count().sort_values(ascending=False).to_string())

Lade WISE C Data ...
wise_data_raw.csv gespeichert
  Zeilen gesamt:   47,995
  Akronyme:        28
  Laender:         119
  Jahre:           [np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]

Zeilen je Akronym:
Acronym
UNDP-HDI                          1904
UNDP-HDI-GNIPC                    1904
WB-WDI-SL-TLF-TOTL-IN             1903
WB-WDI-NY-GDP-PCAP-KD             1886
WB-WDI-NY-GDP-MKTP-KD             1882
UNDP-HDI-GII-LFPR-F               1880
UNDP-HDI-GDI-GNI-PC-M             1880
UNDP-HDI-GDI-GNI-PC-F             1880
UNDP-HDI-GII-LFPR-M               1880
WB-WDI-SL-GDP-PCAP-EM-KD          1869
WB-WDI-NE-CON-GOVT-CD             1814
WB-WDI-FM-AST-NFRG-CN             1802
WB-WDI-BM-KLT-DINV-WD-GD-ZS       1791
WHR-LS                            1781
WB-WDI-TM-VAL-MMTL-ZS-UN    

## Export: Länderliste mit ISO3-Codes und Namen

Fuer die weitere Analyse -- insbesondere fuer das Einlesen und Matching
der weiteren Indikator-Datenbanken in Notebook 02 -- wird eine
Referenzdatei mit allen 119 Studienländern erstellt.

**Spalte A:** ISO3-Code  
**Spalte B:** Ländername (Quelle: World Development Indicators)

In [25]:
# ISO3-Code -> Ländername aus WDI-Metadaten extrahieren
WDI_COUNTRY_PATH = Path('..') / 'Further Indicator DBs' / 'Inflation.xls'
df_wdi_countries = pd.read_excel(
    WDI_COUNTRY_PATH, header=3,
    usecols=['Country Name', 'Country Code']
)
df_wdi_countries = df_wdi_countries.dropna(subset=['Country Code']).copy()
df_wdi_countries['Country Code'] = df_wdi_countries['Country Code'].str.strip()
iso3_to_name = df_wdi_countries.set_index('Country Code')['Country Name'].to_dict()

# Länderliste erstellen (sortiert nach ISO3, wie INCLUDED_COUNTRIES)
country_list = pd.DataFrame({
    'ISO3':         INCLUDED_COUNTRIES,
    'Country_Name': [iso3_to_name[c] for c in INCLUDED_COUNTRIES],
})

# Exportieren
OUTPUT_COUNTRIES = OUTPUT_DIR / 'country_list.xlsx'
country_list.to_excel(OUTPUT_COUNTRIES, index=False)

print(f'Gespeichert: {OUTPUT_COUNTRIES}')
print(f'  Länder: {len(country_list)}')
print()
display(country_list)

Gespeichert: Created Files for Analysis\country_list.xlsx
  Länder: 119



,ISO3,Country_Name
0,AFG,Afghanistan
1,ALB,Albania
2,ARE,United Arab Emirates
3,ARG,Argentina
4,ARM,Armenia
...,...,...
114,VNM,Viet Nam
115,YEM,"Yemen, Rep."
116,ZAF,South Africa
117,ZMB,Zambia
